# 文をクラスタリングして話題ごとにまとめる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/10_sentence_embeddings/sentence_clustering.ipynb)

NLP クックブック 10-04 ／ 文埋め込みとクラスタリング

大量の文（アンケートの自由記述、問い合わせなど）を、意味の近いもの同士で自動的にグループ分けしたい。

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering, KMeans

texts = [
    "アプリがすぐ落ちる", "起動するとクラッシュします", "ログインできません", "パスワードを忘れてログインできない",
    "料金が高すぎる", "月額プランをもっと安くしてほしい", "解約方法がわからない", "退会の手続きを教えてください",
    "動作が重くて使いにくい", "読み込みが遅い", "値段の割に機能が少ない", "サインインの画面でエラーが出る",
]
model = SentenceTransformer("intfloat/multilingual-e5-small")
emb = model.encode(["query: " + t for t in texts], normalize_embeddings=True)

In [2]:
# クラスタ数を決め打ちする場合: k-means
labels = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(emb)
for c in range(4):
    print(c, [t for t, l in zip(texts, labels) if l == c])

0 ['料金が高すぎる', '月額プランをもっと安くしてほしい', '値段の割に機能が少ない']
1 ['アプリがすぐ落ちる', '起動するとクラッシュします', '動作が重くて使いにくい', '読み込みが遅い']
2 ['ログインできません', 'パスワードを忘れてログインできない', 'サインインの画面でエラーが出る']
3 ['解約方法がわからない', '退会の手続きを教えてください']


In [3]:
# クラスタ数がわからない場合: 距離のしきい値で止める階層的クラスタリング
agg = AgglomerativeClustering(n_clusters=None, distance_threshold=0.13, metric="cosine", linkage="average")
labels = agg.fit_predict(emb)
for c in np.unique(labels):
    print(c, [t for t, l in zip(texts, labels) if l == c])

0 ['アプリがすぐ落ちる', '動作が重くて使いにくい', '読み込みが遅い', '値段の割に機能が少ない']
1 ['ログインできません', 'パスワードを忘れてログインできない', '解約方法がわからない']
2 ['起動するとクラッシュします', 'サインインの画面でエラーが出る']
3 ['退会の手続きを教えてください']
4 ['料金が高すぎる', '月額プランをもっと安くしてほしい']


- しきい値はモデルによって適切な値が違う（E5 は全体に類似度が高めに出る）。少量のデータで目視して決める。
- 各クラスタの名前付けは、クラスタ内の代表文（中心に最も近い文）を見るか、LLM に要約させるとよい。